In [0]:
df = spark.read.table("bronze_catalog.default.country_table")
display(df)

In [0]:
df = spark.read.table("bronze_catalog.default.employee_table")
display(df)

In [0]:
bronze_schemapath  = "`bronze_catalog`.default"

In [0]:
tables = spark.catalog.listTables(dbName=bronze_schemapath)

In [0]:
df = spark.read.table("bronze_catalog.default.employee_table")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def fill_id_column(df, column_name):
    w = Window().orderBy(df.columns[0])
    df = df.withColumn(column_name, row_number().over(w))
    return df

In [0]:
from pyspark.sql.functions import to_date, col, split

df = spark.read.table(f"{bronze_schemapath}.employee_table")
df = fill_id_column(df, "employee_id")
df = df.withColumn("hire_date", to_date(col("hire_date"), "dd-MM-yyyy"))
df = df.withColumn("last_update", to_date(col("last_update"), "dd-MM-yyyy"))
df = df.withColumn(
    "employee_internal_id",
    split(col("employee_name"), " ")[1]
)
df = df.dropDuplicates()
display(df)

In [0]:
silver_schemapath = "`silver_catalog`.default"

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.employee_table")